In **K-Nearest Neighbors (KNN)**, the algorithm does **not create groups first** and then measure distance between groups.

It always works like this:

1. Take a **new point** you want to predict.
2. Calculate the distance from that point to **every training point individually**.
3. Pick the **K closest points**.
4. Use those neighbors to decide the output.

So the “distance” is always **between individual data points**, not between clusters/groups.

---

Your example:

Points:

* 2
* 24
* 104
* 150

Suppose:

* (K = 2)

Now imagine a new value comes in:

## Example 1: Predict for 20

Distances from 20:

| Point | Distance |
| ----- | -------- |
| 2     | 18       |
| 24    | 4        |
| 104   | 84       |
| 150   | 130      |

The 2 nearest neighbors are:

* 24
* 2

So KNN uses those two.

---

## Example 2: Predict for 120

Distances:

| Point | Distance |
| ----- | -------- |
| 2     | 118      |
| 24    | 96       |
| 104   | 16       |
| 150   | 30       |

Nearest neighbors:

* 104
* 150

Again, only point-to-point distances matter.

---

## Your main doubt

> “Distance between 104 and 150 is large. Won’t that affect the model?”

Not directly.

KNN does **not care** whether neighbors are far from each other.

It only cares whether they are close to the **query point**.

So if the new point is 120:

* 104 is close
* 150 is also relatively close

Even though 104 and 150 themselves are 46 apart, that does not matter much.

---

## What “weights” are considered?

There are 2 common versions of KNN:

### 1. Uniform weights (default)

Every neighbor gets equal importance.

If (K=2):

[
\text{Prediction} = \frac{\text{neighbor1} + \text{neighbor2}}{2}
]

or for classification:

majority vote.

---

### 2. Distance-weighted KNN

Closer neighbors get more importance.

Typical weight formula:

w_i = \frac{1}{d_i}

where:

* (d_i) = distance to the query point
* closer point ⇒ larger weight

Example for query = 120:

| Neighbor | Distance | Weight |
| -------- | -------- | ------ |
| 104      | 16       | 1/16   |
| 150      | 30       | 1/30   |

So 104 influences the prediction more.

Notice again:

* weight is based on **distance to query point**
* NOT distance between neighbors themselves

---

## Important intuition

KNN assumes:

> “Nearby points in feature space should have similar outputs.”

It does **not** try to form compact clusters like K-Means.

That’s why large gaps between neighbors can sometimes make KNN unstable if:

* data is sparse
* K is too large
* features are not scaled

---

## Another important issue: feature scaling

If one feature has huge values, distances become misleading.

Example:

* Age: 20–50
* Salary: 10,000–1,000,000

Salary dominates the distance calculation.

So KNN usually requires normalization/standardization.

Common distance:

d(x,y)=\sqrt{\sum_{i=1}^{n}(x_i-y_i)^2}

This is Euclidean distance.

---

## Final clarification

KNN does **not** say:

* “group 2–24”
* “group 104–150”

Those are just the nearest points for a specific query.

The model recalculates neighbors separately for every new input.


---------------------------------------------------------------------------------------------------------

Let's walk through the code step by step with a real example.

Suppose you have this data:

```python
dic = {
    'city': ['Kochi', 'Delhi', 'Kochi'],
    'age': [25, 30, 22]
}

feature_list = ['city']
```

Now call:

```python
out = pd.DataFrame(dic)
```

`out` becomes:

| city  | age |
| ----- | --- |
| Kochi | 25  |
| Delhi | 30  |
| Kochi | 22  |

---

## Step 1: Create dummy variables

```python
pd.get_dummies(out[feature_list])
```

Here:

```python
out[feature_list]
```

means:

```python
out[['city']]
```

which is:

| city  |
| ----- |
| Kochi |
| Delhi |
| Kochi |

Now `get_dummies()` converts categories into numeric columns:

```python
pd.get_dummies(out[['city']])
```

Result:

| city_Delhi | city_Kochi |
| ---------- | ---------- |
| 0          | 1          |
| 1          | 0          |
| 0          | 1          |

---

## Step 2: Add dummy columns to original dataframe

```python
out = pd.concat([out, pd.get_dummies(out[feature_list])], axis=1)
```

This combines both side-by-side.

Now `out` becomes:

| city  | age | city_Delhi | city_Kochi |
| ----- | --- | ---------- | ---------- |
| Kochi | 25  | 0          | 1          |
| Delhi | 30  | 1          | 0          |
| Kochi | 22  | 0          | 1          |

Notice:

* Original column `city` still exists
* New dummy columns are added

---

## Step 3: Remove original categorical column

```python
out.drop(feature_list, axis=1, inplace=True)
```

This removes:

```python
['city']
```

Final dataframe:

| age | city_Delhi | city_Kochi |
| --- | ---------- | ---------- |
| 25  | 0          | 1          |
| 30  | 1          | 0          |
| 22  | 0          | 1          |

---

# Why remove the original column?

Machine learning models usually need numbers, not text.

So:

```text
"Kochi", "Delhi"
```

must become:

```text
0 and 1 columns
```

If you keep BOTH:

* original text column
* dummy columns

then the same information is duplicated.

So we:

1. Create numeric columns
2. Delete old text column

---

# Now the second function

Suppose:

## Train data

| age | city_Delhi | city_Kochi |
| --- | ---------- | ---------- |
| 25  | 0          | 1          |
| 30  | 1          | 0          |

---

## Test data

| age | city_Kochi | city_Mumbai |
| --- | ---------- | ----------- |
| 22  | 1          | 0           |

Notice:

* Train has `city_Delhi`
* Test has `city_Mumbai`

Columns are different.

Many ML models require SAME columns in train and test.

So:

```python
common_feat = list(set(train.keys()) & set(test.keys()))
```

finds common columns:

```python
['age', 'city_Kochi']
```

Then:

```python
return train[common_feat], test[common_feat]
```

keeps only common columns.

Final train:

| age | city_Kochi |
| --- | ---------- |
| 25  | 1          |
| 30  | 0          |

Final test:

| age | city_Kochi |
| --- | ---------- |
| 22  | 1          |

Now both have identical columns, so ML model works correctly.
